# 20. OVR ratings

Forecast, Phase 1, step 2e. Every NHL player and prospect gets an **EA NHL-style overall rating (OVR)**, with his projected WAR underneath. The season simulator will use the same scale, so it lives in a small module, `src/ratings.py`.

**The scale** (Ryan's decisions):

- EA-like: NHL regulars mostly 75-95, replacement level about 70, prospects and AHL-level players about 50-70, a floor of 40, the top player about 97-99.
- **One scale for forwards and defensemen:** a 90 forward and a 90 defenseman are worth the same to a team.
- **Goalies get their own anchors**, built by the same rule within goalies (replacement = 70, median regular goalie = 80, top goalie = 95). Section 4 explains why.

**The input is projected WAR/60 from `forecast.current_projections()`**, one table that collects every 2026-27 projection:

- NHL skaters from notebook 18 (player-specific prior)
- NHL goalies and goalie prospects from notebook 21 (goalie prior)
- skater prospects from notebook 19

OVR measures talent, not minutes: a great player on a bad depth role still rates as great.

**Two steps:**

1. **WAR_std:** WAR/60 x the minutes of a full-time role at his position. That puts forwards, defensemen and goalies on the same footing: what would he be worth to a team in a full-time job?
2. **OVR:** WAR_std turned into a 40-99 number through a few fixed anchors (replacement = 70, median regular = 80, top player = 98), with straight lines between them.

**This is the second version of notebook 20.** The first rated goalies on the skater scale with notebook 15's projections, and goalie prospects at a placeholder 70. Now goalies have the notebook 21 prior and their own anchors, and the skater anchors are refrozen from skaters only. **After this the scale is locked** for the season simulator.

Results stay private: `data/ratings_2026-27.csv` and `data/forecast_table_2026-27.csv` (everything the app's Forecast page needs), both gitignored.

## 1. Setup

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
from pathlib import Path

from ratings import war_std, ovr, ovr_exact, STD_MINUTES, ANCHOR_WAR, ANCHOR_OVR, GOALIE_ANCHOR_WAR, GOALIE_ANCHOR_OVR
from forecast import current_projections

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 220)

DATA = Path("../data")
OUT = DATA                      # where ratings_2026-27.csv and forecast_table_2026-27.csv are saved
NAMES = {"F": "Forwards", "D": "Defensemen", "G": "Goalies"}

cur = current_projections(DATA)                               # every NHL player and prospect, one row each
pros = pd.read_csv(DATA / "prospects_2026-27.csv")            # prospect details (draft, league stats; notebook 19)
skaters = pd.read_csv(DATA / "war_by_season_2008_2025.csv")
goalies = pd.read_csv(DATA / "goalie_war_2008_2025.csv")
sk25 = skaters[skaters["season"] == 2025].copy()              # 2025-26 (season = start year)
g25 = goalies[goalies["season"] == 2025].copy()
print(f"Players in current_projections(): {len(cur):,}")
print(cur["source"].value_counts().to_string())
print(f"2025-26 seasons: {len(sk25):,} skaters, {len(g25):,} goalies, {sk25['team'].nunique()} teams")

Players in current_projections(): 2,495
source
NHL skater (nb18)         1196
skater prospect (nb19)    1028
goalie prospect (nb21)     139
NHL goalie (nb21)          132
2025-26 seasons: 940 skaters, 98 goalies, 32 teams


## 2. A standard workload for each position

To compare positions on value, each one gets the minutes of a **full-time role**, measured from 2025-26:

- **Forwards:** each team's top 6 forwards by total ice time (the top two lines), average minutes per game, x 82 games.
- **Defensemen:** each team's top 4 by ice time (the top two pairs), same way.
- **Goalies:** each team's starter (most games played). A starter doesn't play all 82, so his workload is his **games x 60 minutes**.

Why not each player's own minutes? Because OVR should measure talent, not role. A 19-year-old stuck on the fourth line and a star on the top line are both rated as if they had a full-time job.

In [ ]:
sk25["min_per_game"] = sk25["all_toi"] / sk25["games_played"]
rows = []
for g, n in [("F", 6), ("D", 4)]:
    # The n skaters with the most total ice time on each team (a traded player counts for his last team)
    top = sk25[sk25["pos_group"] == g].sort_values("all_toi", ascending=False).groupby("team").head(n)
    mpg = top["min_per_game"].mean()
    rows.append({"group": g, "full-time role": f"top {n} per team by ice time", "players": len(top),
                 "minutes per game": mpg, "games": 82, "median games played": top["games_played"].median(),
                 "standard minutes": mpg * 82})
starters = g25.sort_values("games_played", ascending=False).groupby("team").head(1)
rows.append({"group": "G", "full-time role": "starter (most games per team)", "players": len(starters),
             "minutes per game": 60, "games": starters["games_played"].mean(),
             "median games played": starters["games_played"].median(), "standard minutes": starters["games_played"].mean() * 60})
workload = pd.DataFrame(rows).set_index("group")
print(workload.round(2).to_string())
print(f"\nStarters' games ranged from {starters['games_played'].min()} to {starters['games_played'].max()}.")

# The frozen scale in src/ratings.py was set from these numbers. If the data ever changes, stop here
# instead of silently moving the scale.
for g in ["F", "D", "G"]:
    assert abs(workload.loc[g, "standard minutes"] - STD_MINUTES[g]) < 1, (g, workload.loc[g, "standard minutes"])
print("Frozen STD_MINUTES in ratings.py:", STD_MINUTES, "(match)")

                      full-time role  players  minutes per game  games  median games played  standard minutes
group                                                                                                        
F         top 6 per team by ice time      192             17.84  82.00                 79.0           1462.55
D         top 4 per team by ice time      128             21.35  82.00                 78.0           1751.07
G      starter (most games per team)       32             60.00  49.78                 50.5           2986.88

Starters' games ranged from 33 to 64.
Frozen STD_MINUTES in ratings.py: {'F': 1463, 'D': 1751, 'G': 2987} (match)


## 3. What each player is rated on

- **NHL skaters:** his 2026-27 projection (notebook 18).
- **NHL goalies:** his 2026-27 projection with the goalie prior (notebook 21).
- **Prospect forwards:** his prospect projection (notebook 19), the WAR/60 he'd produce **if he played NHL minutes**.
- **Prospect defensemen:** `d_rank_prior`. Notebook 19 gave every defenseman prospect the same projection (the position average), because no prior beat it on the test rookies. As a rating that would tie all 389 of them at one number, above the median NHL defenseman. The ranking prior spreads them out instead. Their `proj_war_60` (the simulator's input) stays the position average, so **for D prospects the rating and the simulator input differ**.
- **Prospect goalies:** the goalie prior (notebook 21), which replaces the first version's placeholder 70. Notebook 21's rookie test found that goalies who earn 500+ minutes play close to league average, and the prior beat both the league average and replacement level.

The column `rating_war_60` holds what the OVR uses and `rating_note` says which rule applied. `proj_war_60` is always the unchanged projection.

In [ ]:
players = cur.copy()
pros_d = players["prospect"] & (players["group"] == "D")
players["rating_war_60"] = np.where(pros_d, players["d_rank_prior"], players["proj_war_60"])
players["rating_note"] = np.select(
    [~players["prospect"], pros_d, players["group"] == "G"],
    ["NHL projection", "prospect D: d_rank_prior (ranking only)", "prospect G: goalie prior (nb21)"],
    "prospect F: proj_war_60")
assert players["rating_war_60"].notna().all()

# 2025-26 minutes (skaters and goalies), for "regulars" and for the second WAR column
min25 = pd.concat([sk25.set_index("playerId")["all_toi"], g25.set_index("playerId")["toi"]])
gp25 = pd.concat([sk25.set_index("playerId")["games_played"], g25.set_index("playerId")["games_played"]])
players["minutes_2025_26"] = players["playerId"].map(min25).where(~players["prospect"])
players["war_std"] = war_std(players["rating_war_60"], players["group"])
players["regular"] = players["minutes_2025_26"] >= 500         # NHL regulars: 500+ minutes in 2025-26
print(players.groupby(["prospect", "group"]).size().rename("players").to_string())
print(f"\nNHL regulars (500+ minutes in 2025-26): {players['regular'].sum():,}")

prospect  group
False     D        407
          F        789
          G        132
True      D        389
          F        639
          G        139

NHL regulars (500+ minutes in 2025-26): 688


## 4. From WAR_std to OVR: the anchors

Four points, straight lines between them, built the same way for skaters and for goalies:

| WAR_std | Skaters | Goalies | Meaning |
|---|---|---|---|
| 0 | 70 | 70 | **replacement level**: the kind of player a team can call up from the AHL for free (WAR is measured above him) |
| median regular | 80 | 80 | the median NHL regular at that position (500+ minutes in 2025-26) |
| top player | 98 | 95 | the top projected skater / goalie for 2026-27 |
| below 0 | down to 40 | down to 40 | the replacement-to-median slope continues down to a floor of 40 |

Above the top player, the median-to-top slope continues for one more point: a cap of 99 for skaters, 96 for goalies. The anchors are rounded to 0.05 WAR (the top one rounded down, so the top player lands exactly on his anchor).

**Why goalies have their own anchors** (Ryan's decision):

- **The simulator uses WAR, not OVR,** so OVR is only how a player is shown.
- **Goalie projections are far less certain than skaters'.** About 87% of a surprise goalie season is luck (notebook 15), so even the best projected goalie is a much shakier bet than McDavid. The best goalie is shown at 95, just below the best skaters.
- **WAR_std still compares across positions.** The goalie test in notebook 21 found goalie WAR on the same scale as skater WAR, so to compare a goalie with a skater, use WAR_std, not OVR.

**Refreezing the skater anchors.** In the first version of this notebook the median regular (80) included goalies, whose high projections pulled it up. With goalies on their own scale, the skater anchors are recomputed from skaters only (the median regular skater = 80, McDavid = 98).

**The scale is now locked.** These numbers are constants in `src/ratings.py`. The season simulator rates every simulated season with the same constants, so ratings can **rise and fall league-wide** in a multi-year sim: a strong generation can produce more 90s, a weak one fewer. Re-percentiling each season (always the same share of 90s) would hide that, so it's never done.

In [ ]:
def propose_anchors(median, top, top_ovr):
    """Anchors from the median regular and the top player: [floor 40, replacement 70, median 80, top, cap = top + 1]."""
    m = round(median / 0.05) * 0.05                  # median, rounded to 0.05
    t = np.floor(top / 0.05) * 0.05                  # top, rounded down to 0.05
    floor_war = -3 * m                               # 30 OVR points below 70 at the 0 -> median slope (10 per m)
    cap_war = t + (t - m) / (top_ovr - 80)           # one more OVR point at the median -> top slope
    return [round(floor_war, 2), 0.0, round(m, 2), round(float(t), 2), round(float(cap_war), 2)]

nhl_only = players[~players["prospect"]]
anchors = {}
for label, is_g, top_ovr in [("Skaters", False, 98), ("Goalies", True, 95)]:
    grp = nhl_only[(nhl_only["group"] == "G") == is_g]
    median_reg, top = grp.loc[grp["regular"], "war_std"].median(), grp["war_std"].max()
    top_name = grp.loc[grp["war_std"].idxmax(), "name"]
    anchors[label] = propose_anchors(median_reg, top, top_ovr)
    print(f"{label}: {grp['regular'].sum()} regulars, median WAR_std {median_reg:.3f}; top: {top_name}, {top:.3f}")
    a = anchors[label]
    print(f"  anchors (WAR_std -> OVR): " + ", ".join(f"{w:g} -> {o}" for w, o in zip(a, [40, 70, 80, top_ovr, top_ovr + 1])))
    print(f"  slopes: {10 / a[2]:.1f} OVR per WAR between replacement and the median, {(top_ovr - 80) / (a[3] - a[2]):.1f} near the top")

# The first version of notebook 20 used one set of anchors for everyone (goalies included in the median)
OLD_ANCHOR_WAR = [-2.85, 0.00, 0.95, 4.25, 4.43]
print("\nSkater anchors, first version vs. refrozen:")
print(pd.DataFrame({"anchor": ["floor", "replacement", "median regular", "top player", "cap"], "OVR": ANCHOR_OVR,
                    "first version": OLD_ANCHOR_WAR, "refrozen (skaters only)": anchors["Skaters"]}).to_string(index=False))

# The locked scale in ratings.py must equal what the data gives. If the data ever changes, stop here.
assert anchors["Skaters"] == ANCHOR_WAR and ANCHOR_OVR == [40, 70, 80, 98, 99], (anchors["Skaters"], ANCHOR_WAR)
assert anchors["Goalies"] == GOALIE_ANCHOR_WAR and GOALIE_ANCHOR_OVR == [40, 70, 80, 95, 96], (anchors["Goalies"], GOALIE_ANCHOR_WAR)
print("\nLocked anchors in ratings.py match: skaters", ANCHOR_WAR, "| goalies", GOALIE_ANCHOR_WAR)

players["ovr_exact"] = ovr_exact(players["rating_war_60"], players["group"])
players["ovr"] = ovr(players["rating_war_60"], players["group"])

Skaters: 616 regulars, median WAR_std 0.849; top: Connor McDavid, 4.272
  anchors (WAR_std -> OVR): -2.55 -> 40, 0 -> 70, 0.85 -> 80, 4.25 -> 98, 4.44 -> 99
  slopes: 11.8 OVR per WAR between replacement and the median, 5.3 near the top
Goalies: 72 regulars, median WAR_std 2.278; top: Logan Thompson, 4.288
  anchors (WAR_std -> OVR): -6.9 -> 40, 0 -> 70, 2.3 -> 80, 4.25 -> 95, 4.38 -> 96
  slopes: 4.3 OVR per WAR between replacement and the median, 7.7 near the top

Skater anchors, first version vs. refrozen:
        anchor  OVR  first version  refrozen (skaters only)
         floor   40          -2.85                    -2.55
   replacement   70           0.00                     0.00
median regular   80           0.95                     0.85
    top player   98           4.25                     4.25
           cap   99           4.43                     4.44

Locked anchors in ratings.py match: skaters [-2.55, 0.0, 0.85, 4.25, 4.44] | goalies [-6.9, 0.0, 2.3, 4.25, 4.38]


**How many skaters move?** Same projections, first-version anchors vs. refrozen ones. Lowering the median anchor from 0.95 to 0.85 lifts most skaters between replacement and the top.

In [ ]:
sk = players[players["group"] != "G"].copy()
sk["ovr_first"] = np.floor(np.interp(sk["war_std"], OLD_ANCHOR_WAR, ANCHOR_OVR) + 0.5).astype(int)
sk["change"] = sk["ovr"].astype(int) - sk["ovr_first"]
for label, part in [("NHL skaters", sk[~sk["prospect"]]), ("skater prospects", sk[sk["prospect"]])]:
    moved = part["change"] != 0
    print(f"{label}: {moved.sum():,} of {len(part):,} move by 1+ OVR ({moved.mean():.0%}); "
          + ", ".join(f"{c:+d}: {n}" for c, n in part["change"].value_counts().sort_index().items()))
reg_sk = sk[~sk["prospect"] & sk["regular"]]
print(f"Median regular skater: {np.median(np.interp(reg_sk['war_std'], OLD_ANCHOR_WAR, ANCHOR_OVR)):.1f} in the first version, "
      f"{reg_sk['ovr_exact'].median():.1f} now")
print("\nExamples (NHL regulars):")
ex = reg_sk.sort_values("war_std").iloc[np.linspace(0, len(reg_sk) - 1, 9).astype(int)]
print(ex[["name", "team", "position", "war_std", "ovr_first", "ovr", "change"]].round(3).to_string(index=False))

NHL skaters: 524 of 1,196 move by 1+ OVR (44%); -2: 2, -1: 79, +0: 672, +1: 443
skater prospects: 403 of 1,028 move by 1+ OVR (39%); -1: 37, +0: 625, +1: 366
Median regular skater: 78.9 in the first version, 80.0 now

Examples (NHL regulars):
            name team position  war_std  ovr_first  ovr  change
 Barclay Goodrow  SJS        C   -1.124         58   57      -1
   Jeremy Lauzon  VGK        D    0.016         70   70       0
Marcus Johansson  MIN        L    0.302         73   74       1
 Colin Blackwell  DAL        C    0.587         76   77       1
     Ridly Greig  OTT        C    0.848         79   80       1
  Justin Sourdif  WSH        C    1.113         81   81       0
      Troy Terry  ANA        R    1.476         83   83       0
    Martin Necas  COL        C    1.993         86   86       0
  Connor McDavid  EDM        C    4.272         98   98       0


## 5. Distribution checks

Before trusting the scale: where do NHL regulars land? Most of them should be 75-95.

In [ ]:
reg = players[players["regular"]]
bins = list(range(40, 100, 5)) + [100]
hist = pd.cut(reg["ovr"].astype(float), bins, right=False).value_counts().sort_index()
print("NHL regulars by OVR (500+ minutes in 2025-26), each # = 4 players:")
for b, n in hist.items():
    print(f"  {b.left:>2}-{b.right - 1:<2} {n:>4}  {'#' * int(round(n / 4))}")

in_range = reg["ovr"].between(75, 95)
print(f"\nRegulars at 75-95: {in_range.mean():.1%} ({in_range.sum()} of {len(reg)})")
below = reg[reg["ovr"] < 75]
print(f"Below 75: {len(below)} ({len(below) / len(reg):.1%}): " + ", ".join(f"{NAMES[g]} {n}" for g, n in below["group"].value_counts().items()))
print(f"Above 95: {(reg['ovr'] > 95).sum()}")

summary = reg.groupby("group").agg(players=("ovr", "size"), median_war_std=("war_std", "median"),
                                    median_ovr=("ovr_exact", "median"), min_ovr=("ovr", "min"), max_ovr=("ovr", "max"),
                                    ovr_90_plus=("ovr", lambda x: int((x >= 90).sum())))
summary["share of group 90+"] = summary["ovr_90_plus"] / summary["players"]
print("\nBy position (regulars):")
print(summary.rename(index=NAMES).round(2).to_string())

# Goalies on their own scale vs. the skater scale (same projection), to show what the separate anchors change
g_reg = reg[reg["group"] == "G"]
one_scale = np.floor(np.interp(g_reg["war_std"], ANCHOR_WAR, ANCHOR_OVR) + 0.5)
print(f"\nRegular goalies: median OVR {g_reg['ovr_exact'].median():.1f} on the goalie scale, {np.median(one_scale):.0f} on the skater scale; "
      f"90+: {(g_reg['ovr'] >= 90).sum()} vs. {(one_scale >= 90).sum()} of {len(g_reg)}")

NHL regulars by OVR (500+ minutes in 2025-26), each # = 4 players:
  40-44    0  
  45-49    0  
  50-54    0  
  55-59    2  
  60-64   13  ###
  65-69   50  ############
  70-74  112  ############################
  75-79  152  ######################################
  80-84  229  #########################################################
  85-89   93  #######################
  90-94   32  ########
  95-99    5  #

Regulars at 75-95: 73.8% (508 of 688)
Below 75: 177 (25.7%): Forwards 88, Defensemen 87, Goalies 2
Above 95: 3

By position (regulars):
            players  median_war_std  median_ovr  min_ovr  max_ovr  ovr_90_plus  share of group 90+
group                                                                                             
Defensemen      215            0.50       75.92       61       92            1                 0.0
Forwards        401            1.09       81.27       57       98           33                0.08
Goalies          72            2.28       79.90   

**Reading this:**

- **Defensemen rate lower than forwards on one scale.** The WAR model credits a defenseman less per full-time role than a forward, so the median defenseman sits below the median forward. That's what "one scale" means: a 90 defenseman really is worth as much as a 90 forward, there are just fewer of them.
- **Goalies are spread around 80 by construction.** Their anchors put the median regular goalie at 80 and the top goalie at 95. On the skater scale the same projections would cluster high (a league-average goalie on a starter's workload is worth a lot more than replacement); notebook 21 found that cluster is the real value of the job, and WAR_std still shows it.

## 6. Hockey checks

The top of the scale, by position, the Red Wings, the prospects, and the bottom of the NHL.

`war_2025_minutes` (the second WAR column) is his projected WAR at **his own** 2025-26 workload: minutes per game x 82 for skaters, total 2025-26 minutes for goalies (minutes per game x 82 would mean starting every game). It's blank for prospects and for players who didn't play in 2025-26.

In [ ]:
mpg25 = (players["playerId"].map(min25) / players["playerId"].map(gp25))
own_minutes = np.where(players["group"] == "G", players["minutes_2025_26"], mpg25 * 82)
players["war_2025_minutes"] = (players["proj_war_60"] * own_minutes / 60).where(players["minutes_2025_26"].notna() & ~players["prospect"])

SHOW = ["name", "team", "position", "age", "proj_war_60", "war_std", "ovr", "war_2025_minutes", "minutes_2025_26"]
nhl_only = players[~players["prospect"]]
print("Top 25 overall by OVR (goalies on their own scale):")
print(nhl_only.nlargest(25, "ovr_exact")[SHOW].round(3).to_string(index=False))

Top 25 overall by OVR (goalies on their own scale):
             name team position  age  proj_war_60  war_std  ovr  war_2025_minutes  minutes_2025_26
   Connor McDavid  EDM        C 30.0        0.175    4.272   98             5.506         1885.450
  Auston Matthews  TOR        C 29.0        0.171    4.179   98             4.873         1248.233
   Leon Draisaitl  EDM        C 31.0        0.162    3.941   96             4.766         1402.383
   Logan Thompson  WSH        G 29.0        0.086    4.288   95             4.945         3445.050
    Cole Caufield  MTL        R 26.0        0.148    3.612   95             3.681         1472.867
   David Pastrnak  BOS        R 30.0        0.145    3.539   94             4.098         1590.483
 Nathan MacKinnon  COL        C 31.0        0.143    3.495   94             4.360         1780.950
Macklin Celebrini  SJS        C 20.0        0.139    3.380   93             4.038         1747.900
  Nikita Kucherov  TBL        R 33.0        0.138    3.35

In [ ]:
for g in ["F", "D", "G"]:
    print(f"\nTop 10 {NAMES[g].lower()}:")
    print(nhl_only[nhl_only["group"] == g].nlargest(10, "ovr_exact")[SHOW].round(3).to_string(index=False))
print("\nTop 10 by WAR_std, all positions (the cross-position comparison):")
print(nhl_only.nlargest(10, "war_std")[SHOW].round(3).to_string(index=False))


Top 10 forwards:
             name team position  age  proj_war_60  war_std  ovr  war_2025_minutes  minutes_2025_26
   Connor McDavid  EDM        C 30.0        0.175    4.272   98             5.506         1885.450
  Auston Matthews  TOR        C 29.0        0.171    4.179   98             4.873         1248.233
   Leon Draisaitl  EDM        C 31.0        0.162    3.941   96             4.766         1402.383
    Cole Caufield  MTL        R 26.0        0.148    3.612   95             3.681         1472.867
   David Pastrnak  BOS        R 30.0        0.145    3.539   94             4.098         1590.483
 Nathan MacKinnon  COL        C 31.0        0.143    3.495   94             4.360         1780.950
Macklin Celebrini  SJS        C 20.0        0.139    3.380   93             4.038         1747.900
  Nikita Kucherov  TBL        R 33.0        0.138    3.358   93             3.827         1545.467
      Jack Hughes  NJD        C 25.0        0.131    3.191   92             3.812         1

In [ ]:
det = nhl_only[nhl_only["team"] == "DET"].sort_values("ovr_exact", ascending=False)
print("Detroit Red Wings (last team in our data; players who didn't play in 2025-26 have no 2025-26 minutes):")
print(det[SHOW].round(3).to_string(index=False))
det_p = players[players["prospect"] & (players["team"] == "DET")].sort_values("ovr_exact", ascending=False)
print(f"\nRed Wings prospects ({len(det_p)}):")
print(det_p[["name", "position", "age", "rating_war_60", "war_std", "ovr", "rating_note"]].round(3).to_string(index=False))

Detroit Red Wings (last team in our data; players who didn't play in 2025-26 have no 2025-26 minutes):
                   name team position  age  proj_war_60  war_std  ovr  war_2025_minutes  minutes_2025_26
         Alex DeBrincat  DET        R 29.0        0.119    2.906   91             3.013         1517.017
          Lucas Raymond  DET        L 24.0        0.114    2.790   90             2.940         1503.917
           Dylan Larkin  DET        C 30.0        0.097    2.360   88             2.670         1493.850
           Patrick Kane  DET        R 38.0        0.061    1.497   83             1.484         1185.583
            John Gibson  DET        G 33.0        0.054    2.692   83             2.867         3181.233
          Moritz Seider  DET        D 25.0        0.047    1.367   83             1.643         2104.650
     James van Riemsdyk  DET        L 37.0        0.048    1.178   82             0.776          846.167
          Emmitt Finnie  DET        C 21.0        0.043  

**Prospects.** A prospect's rating means "if he played NHL minutes this season". Whether he does is Phase 2's call-up logic, not a rating discount. Defensemen are ranked by `d_rank_prior`; their `proj_war_60` is the shared position average. Goalie prospects use the notebook 21 goalie prior.

In [ ]:
pinfo = pros.set_index("playerId")[["draft_year", "pick", "league_2025_26", "gp_2025_26", "points_2025_26"]]
pp = players[players["prospect"]].join(pinfo, on="playerId")
PSHOW = ["name", "position", "age", "team", "draft_year", "pick", "league_2025_26", "gp_2025_26", "points_2025_26",
         "proj_war_60", "d_rank_prior", "war_std", "ovr"]
print("Top 15 prospects by OVR:")
print(pp.nlargest(15, "ovr_exact")[PSHOW].round(3).to_string(index=False))
print("\nProspect OVR by group:")
print(pp.groupby("group")["ovr"].describe()[["count", "min", "25%", "50%", "75%", "max"]].rename(index=NAMES).to_string())
print("\nTop 5 defenseman prospects (by d_rank_prior):")
print(pp[pp["group"] == "D"].nlargest(5, "ovr_exact")[PSHOW].round(3).to_string(index=False))
gpp = pp[pp["group"] == "G"]
print(f"\nGoalie prospects: {len(gpp)}, median OVR {gpp['ovr_exact'].median():.1f} (range {gpp['ovr'].min()}-{gpp['ovr'].max()}); top 5:")
print(gpp.nlargest(5, "ovr_exact")[PSHOW].round(3).to_string(index=False))

Top 15 prospects by OVR:
               name position  age team  draft_year  pick league_2025_26  gp_2025_26  points_2025_26  proj_war_60  d_rank_prior  war_std  ovr
      Gavin McKenna        L 19.0  TOR        2026     1           NCAA        35.0            51.0        0.100           NaN    2.434   88
    Roman Kantserov        R 22.0  CHI        2023    44            KHL        63.0            64.0        0.094           NaN    2.288   88
    Andrew Cristall        L 21.0  WSH        2023    40            AHL        72.0            60.0        0.078           NaN    1.891   86
       Kim Saarinen        G 20.0  SEA        2024    88          Liiga        38.0             0.0        0.058           NaN    2.895   85
   Ethan Wyttenbach        L 19.0  CGY        2025   144           NCAA        40.0            59.0        0.070           NaN    1.709   85
    Ryder Fetterolf        G 19.0  CAR        2026   125            OHL        41.0             0.0        0.058           NaN   

In [ ]:
# Depth players: forwards who played 40+ games at under 12 minutes a night in 2025-26 (fourth-liners)
fourth = nhl_only[(nhl_only["group"] == "F") & (nhl_only["playerId"].map(gp25) >= 40) & (mpg25.reindex(nhl_only.index) < 12)]
print(f"Fourth-line forwards (40+ games, under 12 min/game in 2025-26): {len(fourth)}, "
      f"median OVR {fourth['ovr_exact'].median():.1f} (25th-75th percentile {fourth['ovr_exact'].quantile(0.25):.1f}-{fourth['ovr_exact'].quantile(0.75):.1f})")
print(fourth.sample(8, random_state=1).sort_values("ovr_exact", ascending=False)[SHOW].round(3).to_string(index=False))

# Third-pair defensemen: 40+ games at under 17 minutes a night
third = nhl_only[(nhl_only["group"] == "D") & (nhl_only["playerId"].map(gp25) >= 40) & (mpg25.reindex(nhl_only.index) < 17)]
print(f"\nThird-pair defensemen (40+ games, under 17 min/game): {len(third)}, median OVR {third['ovr_exact'].median():.1f}")

print("\nReplacement-level examples (regulars rated 70, most 2025-26 minutes first):")
print(nhl_only[nhl_only["regular"] & (nhl_only["ovr"] == 70)].nlargest(6, "minutes_2025_26")[SHOW].round(3).to_string(index=False))
print("\nLowest-rated regulars:")
print(nhl_only[nhl_only["regular"]].nsmallest(5, "ovr_exact")[SHOW].round(3).to_string(index=False))

Fourth-line forwards (40+ games, under 12 min/game in 2025-26): 77, median OVR 73.3 (25th-75th percentile 69.8-77.5)
            name team position  age  proj_war_60  war_std  ovr  war_2025_minutes  minutes_2025_26
      Max Sasson  VAN        C 26.0        0.039    0.949   81             0.623          773.450
  Jesper Boqvist  FLA        C 28.0        0.033    0.812   80             0.538          863.517
Hendrix Lapierre  WSH        C 24.0        0.025    0.607   77             0.300          653.717
  Jansen Harkins  ANA        C 29.0        0.020    0.477   76             0.257          421.983
  Ozzy Wiesblatt  NSH        C 24.0        0.001    0.036   70             0.021          407.283
    Kyle MacLean  NYI        C 27.0       -0.000   -0.001   70            -0.001          620.650
     Jeff Malott  LAK        L 30.0       -0.009   -0.215   67            -0.106          511.500
   Rodrigo Abols  PHI        C 31.0       -0.014   -0.333   66            -0.196          441.033



## 7. Save

- **`data/ratings_2026-27.csv`:** one row per NHL player and prospect, with the projection, the WAR/60 the rating uses, WAR_std (**WAR per 82 games in a full-time role**; for goalies, per starter's season), OVR and the WAR at his 2025-26 minutes.
- **`data/forecast_table_2026-27.csv`:** everything the app's Forecast page shows, in one table. It has the same rows, plus `source`, 2025-26 minutes and, for prospects, draft details, 2025-26 league stats, NHLe and the p10-p90 range (skater prospects only). For D prospects the range belongs to the projection (the position average), not to the `d_rank_prior` rating.

Both are gitignored. The app reads only the forecast table, and only locally.

In [ ]:
COLS = ["playerId", "name", "team", "position", "group", "age", "prospect", "proj_war_60", "rating_war_60", "rating_note",
        "war_std", "ovr", "war_2025_minutes"]
out = players[COLS].sort_values("ovr", ascending=False, kind="stable")
assert out["playerId"].is_unique and len(out) == len(cur)
out.to_csv(OUT / "ratings_2026-27.csv", index=False)
print(f"Saved ratings_2026-27.csv: {len(out):,} players ({(~out['prospect']).sum():,} NHL, {out['prospect'].sum():,} prospects)")

# The Forecast page's table: the ratings plus prospect details
details = pros.set_index("playerId")[["draft_year", "pick", "round", "draft_team", "league_2025_26", "gp_2025_26",
                                      "points_2025_26", "nhle_last2"]]
ft = players.join(details, on="playerId")
ft["war_std_p10"] = war_std(ft["war_60_p10"], ft["group"])
ft["war_std_p90"] = war_std(ft["war_60_p90"], ft["group"])
FT_COLS = ["playerId", "name", "team", "position", "group", "age", "prospect", "source", "played_2025_26",
           "ovr", "war_std", "proj_war_60", "rating_war_60", "rating_note", "war_2025_minutes", "minutes_2025_26",
           "draft_year", "pick", "round", "draft_team", "league_2025_26", "gp_2025_26", "points_2025_26", "nhle_last2",
           "d_rank_prior", "war_60_p10", "war_60_p90", "war_std_p10", "war_std_p90"]
ft = ft[FT_COLS].sort_values(["ovr", "war_std"], ascending=False, kind="stable")
assert ft["playerId"].is_unique and len(ft) == len(cur)
assert ft.loc[ft["prospect"], "draft_year"].notna().all()
ft.to_csv(OUT / "forecast_table_2026-27.csv", index=False)
print(f"Saved forecast_table_2026-27.csv: {len(ft):,} rows, {len(FT_COLS)} columns")

Saved ratings_2026-27.csv: 2,495 players (1,328 NHL, 1,167 prospects)
Saved forecast_table_2026-27.csv: 2,495 rows, 29 columns


## Takeaways
- **The scale is locked in `src/ratings.py`.**
  - **Standard workloads from 2025-26 (unchanged):** forwards 17.84 min/game (top 6 per team) = 1,463 minutes; defensemen 21.35 (top 4) = 1,751; goalies 49.78 starts (33 to 64) x 60 = 2,987.
  - **Skater anchors (refrozen, skaters only):** WAR_std 0 = 70 (replacement), 0.85 = 80 (median of 616 regular skaters, 0.849), 4.25 = 98 (McDavid, 4.272), floor 40 at -2.55, cap 99 at 4.44.
  - **Goalie anchors (new):** 0 = 70, 2.30 = 80 (median of 72 regular goalies, 2.278), 4.25 = 95 (Logan Thompson, 4.288), floor 40 at -6.90, cap 96 at 4.38.
  - **What 1 WAR is worth:**
    - skaters: 11.8 OVR points between replacement and the median, 5.3 near the top
    - goalies: 4.3 and 7.7
  - The simulator rates every simulated season with these constants, so ratings can rise and fall league-wide; nothing is re-percentiled.
- **Goalies are on their own OVR scale; WAR_std still compares across positions.**
  - The simulator uses WAR, and goalie projections are far less certain (about 87% of a surprise goalie season is luck), so the best goalie shows at 95, below the best skaters.
  - On WAR_std, Thompson (4.29) is still the top player in the league, just ahead of McDavid (4.27). Shesterkin and Hellebuyck (3.79, 3.78) are 5th and 6th.
  - **Regular goalies:** 72 to 95 on the goalie scale (median 79.9), and 3 of 72 are 90+. On the skater scale the same projections would have a median of 88 and 24 goalies at 90+.
- **The refreeze moves skaters by at most a point or two, mostly up.**
  - **NHL skaters:** 524 of 1,196 move by 1+ OVR (44%): 443 up 1, 79 down 1, 2 down 2.
  - **Skater prospects:** 403 of 1,028 move: 366 up 1, 37 down 1.
  - **Why:** the median regular skater goes from 78.9 to 80.0, because goalies no longer pull the median anchor up (0.95 to 0.85). Players below replacement fall slightly, because the steeper lower slope continues down to the floor.
- **Most regulars land 75-95:** 73.8% (508 of 688).
  - 177 are below 75: 88 forwards, 87 defensemen and 2 goalies.
  - Only 3 regulars are above 95: McDavid and Matthews 98, Draisaitl 96.
- **The top makes hockey sense.**
  - Forwards: McDavid and Matthews 98, Draisaitl 96, Caufield 95, Pastrnak and MacKinnon 94, Celebrini 93 at 20.
  - Makar (92) is the top defenseman, then Dahlin, Fox, Werenski and Quinn Hughes at 89.
- **Top goalies:** Thompson 95, then Shesterkin and Hellebuyck 91, Hart and Swayman 89, Sorokin and Vasilevskiy 88.
  - **Flag:** the expectation was roughly 93-95 for the top three. Following the rule as written (median regular goalie = 80, top = 95), the median goalie already sits at 2.30 WAR_std. That leaves only 1.95 WAR between 80 and 95, so a goalie 0.5 WAR behind Thompson is 4 points lower.
  - The anchors were not tuned to hit those numbers.
- **Defensemen still rate lower than forwards on one skater scale.** The median regular is 75.9 for defensemen vs. 81.3 for forwards. Only 1 of 215 regular defensemen is 90+, vs. 33 of 401 forwards.
  - **Flag (unchanged):** heavy-minute defensemen the model sees as replacement level still rate 70 (Trouba, DeMelo, York, Borgen).
- **Prospects are rated "if he plays NHL minutes".**
  - **Forwards:** 62 to 88 (median 75). Gavin McKenna and Roman Kantserov are 88, Andrew Cristall 86.
  - **Defensemen:** use `d_rank_prior`, 67 to 80 (median 71; Henry Mews, Eric Pohlkamp and Bryce Pickford 80). Their `proj_war_60` stays the position average.
  - **Goalies:** now use the notebook 21 goalie prior instead of a placeholder 70: 74 to 85, median 79 (expected near 80). Kim Saarinen and Ryder Fetterolf are 85.
  - Readiness belongs in Phase 2's call-up logic, not in a rating discount.
- **The bottom of the NHL:**
  - Fourth-line forwards (77 with 40+ games under 12 min/game): median 73.3 (25th-75th percentile 69.8-77.5).
  - Third-pair defensemen (49): median 74.4.
  - The lowest regulars are Goodrow and Ross Johnston, both 57.
- **Red Wings:**
  - NHL roster: DeBrincat 91, Raymond 90, Larkin 88; then Kane, Gibson and Seider 83; Sandin-Pellikka 80. The other goalies are Cossa 79, Hutchinson 76 and Talbot 75. Chiarot (61) and Hamonic (60) are lowest.
  - Prospects (48): Guimond (G), Max Plante and Buchelnikov 84; Augustine (G) and Lombardi 82.
- **Files:**
  - `ratings_2026-27.csv`: same columns as before, on the locked scale.
  - `forecast_table_2026-27.csv`: the app's Forecast page reads only this file, locally. It has 2,495 rows and 29 columns: ratings, source, 2025-26 minutes, and for prospects draft details, 2025-26 stats, NHLe and the p10-p90 range.
  - Both are gitignored.
- **Next (Phase 2):** minutes projections (who plays, how much, and which prospects get called up), then the season simulator, which imports `ratings.py` and turns team WAR into goals with fitted coefficients (notebook 21).